In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
assistant.rag("How do I run Ollama locally?")


'To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download for your operating system.\n2. Open a terminal and run:\n\n```bash\nollama run llama3\n```\n\nThis downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.\n\nTo test that the local server is running, you can also run:\n\n```bash\ncurl http://localhost:11434\n```\n\nIf you get a `Connection refused` error while prompting Ollama during the homework, restart the Ollama server with:\n\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```'

In [5]:
assistant.rag("How do I run Olama locally?")

'I don’t have any FAQ entry about **Olama** specifically.\n\nIf you mean running the **course locally**, the FAQ says you can do that if you’re comfortable setting up **Python, `uv`, Jupyter, Docker, and any other tools needed for the module**. It also says to **document your setup and keep it reproducible**.\n\nIf you meant a different tool name, tell me which one and I’ll look for the matching FAQ guidance.'

In [6]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—often you can, but it depends on the course’s enrollment rules and whether it’s still open.\n\nIf you want, I can help you figure it out. Please send me:\n- the course name or link\n- where it’s hosted\n- whether it’s live or self-paced\n- and, if known, the start date\n\nThen I can tell you the likely next step and help you draft a message to the instructor or support team if needed.'

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [9]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I still join"}', call_id='call_itliCzlDiG7EE6TSERhB7apf', name='search', type='function_call', id='fc_0f1f1f627788d429006a3bcfd2021c8199802ea0cfa33946c4', namespace=None, status='completed')]

In [10]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [11]:
result_json

'[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "9f689c185f",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I missed the first homework - can I still get a certificate?",\n    "answer": "Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you 

In [12]:
messages.extend(response.output)
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [14]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'

In [15]:
print(messages)

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}, ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I still join"}', call_id='call_itliCzlDiG7EE6TSERhB7apf', name='search', type='function_call', id='fc_0f1f1f627788d429006a3bcfd2021c8199802ea0cfa33946c4', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_itliCzlDiG7EE6TSERhB7apf', 'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "9f689c185f",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I missed the first homework - can I still get a certificate?",\n    "answer": "Yes, you ne

In [16]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(810, 29)

In [17]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(810, 29)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001389
